# Laya × Integrated Memory V2.2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_Integrated_Memory_V22_Colab.ipynb)

Bidirectional Integrated Memory V2.2 adaptation: pretrained ModernBERT Wqkv/Wo stay frozen; token mixing is replaced by low-rank global kernel memory + local CeNN-style depthwise mixing + direct-value path.

**Goal.** Replace selected attention blocks in 'convaiinnovations/laya' without changing Laya's tokenizer, typed decision head, calibration/runtime API, or 'Router'. The replacement is trained by attention-output distillation from the untouched Laya teacher and is accepted **one layer at a time** only when local fidelity *and* Laya decision-level gates pass.

Because Laya uses **bidirectional ModernBERT**, these are encoder-specific adaptations of the TinyCeNN methods. They are experiments; a passed causal-LM result is not assumed to transfer automatically.


## 1. Setup
A T4/L4/A100 runtime is recommended. The notebook installs the latest Laya source plus this TinyCeNN-LM repository.


In [ ]:
import os, sys, subprocess, pathlib, importlib
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Works in Colab, Kaggle, and ordinary Jupyter kernels.
if pathlib.Path("/content").exists():
    WORK = pathlib.Path("/content")
elif pathlib.Path("/kaggle/working").exists():
    WORK = pathlib.Path("/kaggle/working")
else:
    WORK = pathlib.Path.cwd()

REPO = WORK / "TinyCeNN-LM"
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "pull", "-q"])

# IMPORTANT: install with this notebook kernel's Python, not a possibly different `pip` executable.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NandhaKishorM/laya.git", "datasets", "pandas", "pyarrow", "safetensors"])

# Editable-install fallback for notebook environments that cache import paths.
SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import tinycenn_lm
from tinycenn_lm.laya_lab import LayaLabConfig, run_experiment
import torch, json, pandas as pd
print("TinyCeNN-LM:", pathlib.Path(tinycenn_lm.__file__).resolve())
print("torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. Experiment configuration
'balanced' is the default practical run. Use 'smoke' to verify the pipeline quickly or 'extended' for a stronger experiment.

The default target is exactly 'convaiinnovations/laya'. To test the fine-tuned typed-decisions checkpoint with the same replacement method, change 'MODEL_ID' to 'convaiinnovations/laya-typed-decisions'.


In [ ]:

MODEL_ID = "convaiinnovations/laya" #@param ["convaiinnovations/laya", "convaiinnovations/laya-typed-decisions"]
MODE = "balanced" #@param ["smoke", "balanced", "extended"]

cfg = LayaLabConfig(
    architecture="integrated_memory_v22",
    model_id=MODEL_ID,
    mode=MODE,
    seed=2026,
    output_dir="/content/laya_tinycenn",
)
print(cfg)


## 3. Train with strict sequential acceptance
Candidate ModernBERT attention layers are tried one at a time. A replacement is kept only if it passes:

- attention-output NMSE and cosine fidelity;
- teacher top-1 decision agreement;
- teacher→student probability KL;
- maximum allowed accuracy drop on held-out Laya-style typed decisions.

Rejected layers are restored to original ModernBERT attention.


In [ ]:
teacher, student, report = run_experiment(cfg)


## 4. Laya-native result summary
The final report emphasizes what matters for Laya: typed decision accuracy, soft-label agreement, Brier/ECE, score MAE, teacher agreement, and request latency.


In [ ]:
summary = pd.DataFrame([
    {"model":"Laya teacher", **{k: report["teacher_final"].get(k) for k in ["accuracy","soft_accuracy","brier","brier_vs_soft","ece","score_mae","ms_per_case"]}},
    {"model":report["architecture"], **{k: report["student_final"].get(k) for k in ["accuracy","soft_accuracy","brier","brier_vs_soft","ece","score_mae","ms_per_case"]},
     "teacher_agreement": report["student_final"].get("teacher_agreement"),
     "teacher_KL": report["student_final"].get("mean_teacher_kl")},
])
display(summary)
print("Accepted attention layers:", report["accepted_layers"])
print("Replacement trainable parameters:", f'{report["replacement_trainable_parameters"]:,}')
print("Latency:", json.dumps(report["latency"], indent=2))

for primitive, metrics in report["student_final"]["by_type"].items():
    print(primitive, metrics)


## 5. Re-run your Laya example on the adapted model
The same 'Router' API is preserved. We attach the already-loaded adapted Agent so there is no second model copy. The architecture here replaces the **English Laya** encoder only; Laya's multilingual route can remain the original multilingual checkpoint.


In [ ]:
from laya import Router

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else"
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"]
    },
    "churn_risk": {"type": "noul", "instructions": "Does the user threaten to cancel or leave?"},
    "refund_requested": {"type": "noul", "instructions": "Does the user explicitly request a refund?"}
}

route_name = "typed-decisions" if "typed-decisions" in MODEL_ID else "english"
router = Router(preload=False)
router.attach(route_name, student)
res = router.predict(state, questions, model=route_name)

print("Department       :", res["answers"]["department"]["choice"])
print("Urgency score    :", res["answers"]["urgency"]["score"])
print("Churn risk       :", res["answers"]["churn_risk"]["noul"])
print("Refund requested :", res["answers"]["refund_requested"]["noul"])
print("Routing          :", res["routing"]["model"])
print("\nTeacher/student raw demo comparison:")
print(json.dumps(report["demo"], indent=2, ensure_ascii=False))


## 6. Inspect layer-by-layer acceptance
A model is not declared successful merely because training loss decreases. This table shows which proposed attention replacements survived the decision-level quality gates.


In [ ]:
rows=[]
for h in report["history"]:
    rows.append({
        "layer":h["layer"], "attention_type":h["attention_type"], "accepted":h["accepted"],
        "nmse":h["local"]["nmse"], "cosine":h["local"]["cosine"],
        "teacher_agreement":h["gate"].get("teacher_agreement"),
        "mean_teacher_kl":h["gate"].get("mean_teacher_kl"),
        "accuracy":h["gate"].get("accuracy"), "accuracy_drop":h["accuracy_drop"],
    })
display(pd.DataFrame(rows))


## 7. Saved outputs
The notebook writes an adapter-only PyTorch checkpoint plus a JSON report under '/content/laya_tinycenn/<architecture>/'. The report contains the acceptance history, official Laya-style evaluation, latency measurements, and the duplicate-charge demo.


In [ ]:
from pathlib import Path
out = Path(cfg.output_dir) / cfg.architecture
print("Adapter:", out / "adapter.pt")
print("Report :", out / "report.json")
print((out / "report.json").read_text()[:4000])
